# Session 22: Language models and building with them

Build a word-matching search over the eight Lakeside Mart policies, test it on twelve questions with a failures column, tune it, then try the sentence-transformers and small model cells in Colab.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Split the policies into passages
Read each file in `policies`, cut it into sentences and keep the file name with each. Store the list in `passages`.

In [ ]:
import os, re
import numpy as np
from collections import Counter

passages = []
for name in sorted(os.listdir('policies')):
    text = ' '.join(open('policies/' + name).read().split())
    for s in re.split(r'(?<=\.)\s+(?=[A-Z])', text):
        passages.append((name, s))

In [ ]:
assert len(passages) == 59, 'Expected 59 passages: split after a full stop and a capital letter'
assert passages[28] == ('loyalty_programme.txt', 'Points expire after 12 months.'), 'Passage 28 should be the loyalty sentence about points expiring. Keep the files in sorted order and the file name with each sentence'
print('Task 1 correct')

## Task 2. Search by shared words
Build `retrieve(question)` that scores each passage by the words it shares with the question (ignore the filler words) and returns the best passage and its score.

In [ ]:
FILLER = set('the a an of to in is are do does how what '
  'can my for on at and or it be by with if we you from '
  'when which will i'.split())
def words(text):
    w = re.findall('[a-z0-9]+', text.lower())
    return {x for x in w if x not in FILLER}
def retrieve(question):
    q = words(question)
    sc = [len(q & words(p)) for n, p in passages]
    i = int(np.argmax(sc))
    return passages[i], sc[i]

In [ ]:
(name, sentence), s = retrieve('How many days do I have to return an item?')
assert name == 'returns_policy.txt', 'The returns policy should win this question. Check words() and that retrieve returns the passage with the most shared words'
print('Task 2 correct')

## Task 3. Build the test sheet
Write twelve test questions as a table `tests` with columns `question`, `policy` and `answer`. Each `answer` must be copied word for word from the policy file. Starter rows are given. Add the missing ones from the files yourself.

In [ ]:
import pandas as pd

tests = pd.DataFrame([
    ('How long do points last?', 'loyalty_programme.txt', 'Points expire after 12 months'),
    ('How many days do I have to return an item?', 'returns_policy.txt', 'within 7 days of purchase'),
    ('How long do mobile money refunds take?', 'returns_policy.txt', 'up to 3 working days'),
    ('Which towns does delivery not reach?', 'delivery.txt', 'Gulu, Mbale, Mbarara or Fort Portal'),
    ('What is the delivery fee within 5 kilometres?', 'delivery.txt', 'UGX 5,000 within 5 kilometres'),
    ('What percent discount do Gold customers get?', 'loyalty_programme.txt', 'Gold customers receive 5 percent off'),
    ('How long is customer data kept?', 'data_privacy.txt', '3 years after the last purchase'),
    ('What temperature must dairy be stored at?', 'food_safety.txt', 'below 5 degrees Celsius'),
    ('When do branches open on Sunday?', 'opening_hours.txt', 'from 9 am to 6 pm on Sunday'),
    ('What happens when more than 5 percent of a delivery is damaged?', 'supplier_complaints.txt', 'the delivery is rejected'),
    ('Does Lakeside Mart ask for my PIN?', 'mobile_money.txt', "never asks for a customer's PIN"),
    ('Can points be exchanged for cash?', 'loyalty_programme.txt', 'Points cannot be exchanged for cash'),
], columns=['question', 'policy', 'answer'])
print(len(tests), 'questions')

In [ ]:
assert len(tests) == 12, 'tests needs twelve rows. Add six more questions with their policy file and answer'
assert tests.policy.nunique() >= 6, 'Cover at least six different policy files'
for _, t in tests.iterrows():
    text = ' '.join(open('policies/' + t.policy).read().split())
    assert t.answer in text, 'Answer not found word for word in ' + t.policy + ': ' + t.answer
print('Task 3 correct')

## Task 4. Score the simple method
Write `score(tests, finder)` that runs each question through a finder and returns a table with the policy it got, whether the right sentence was found, and a `failure` column.

In [ ]:
def score(tests, finder):
    rows = []
    for _, t in tests.iterrows():
        (name, sentence), s = finder(t.question)
        right = name == t.policy
        found = t.answer in sentence
        if right and found:
            fail = ''
        elif right:
            fail = 'right policy, wrong sentence'
        else:
            fail = 'wrong policy'
        rows.append((t.question, t.policy, name, found, fail))
    return pd.DataFrame(rows, columns=['question', 'policy', 'got', 'answer_found', 'failure'])

r1 = score(tests, retrieve)
print((r1.policy == r1.got).sum(), 'of 12 right policy;', r1.answer_found.sum(), 'answers found')
r1[r1.failure != ''][['question', 'got', 'failure']]

In [ ]:
assert (r1.policy == r1.got).sum() >= 9, 'At least 9 of 12 should pick the right policy. Check your word sets.'
assert set(r1.failure) <= {'', 'wrong policy', 'right policy, wrong sentence'}
print('Task 4 correct')

## Task 5. Tune the method
Read the failures. Add a short list of words that mean the same, cut a trailing s, and weight rare words more. Build `retrieve2` and score it again. Your tuned method must pick the right policy for at least 11 of 12 questions and find at least 11 answers.

In [ ]:
SAME = {'last': 'expire', 'lasts': 'expire'}
def base(w):
    w = SAME.get(w, w)
    return w[:-1] if len(w) > 3 and w[-1] == 's' else w
words2 = lambda text: {base(w) for w in words(text)}
sets = [words2(p) for n, p in passages]
seen_in = Counter(w for s in sets for w in s)
def retrieve2(question):
    q = words2(question)
    sc = [sum(1 / seen_in[w] for w in q & s) for s in sets]
    i = int(np.argmax(sc))
    return passages[i], round(sc[i], 2)

r2 = score(tests, retrieve2)
print((r2.policy == r2.got).sum(), 'of 12 right policy;', r2.answer_found.sum(), 'answers found')

In [ ]:
assert (r2.policy == r2.got).sum() >= 11, 'Fewer than 11 right policies. Look at the failure column for r2 and fix the cause'
assert r2.answer_found.sum() >= 11, 'Fewer than 11 answers found. Look at the failure column for r2'
assert (r2.policy == r2.got).sum() >= (r1.policy == r1.got).sum(), 'The tuned method should not do worse than the simple one'
print('Task 5 correct')

## Task 6. Search by meaning (Colab only, needs internet)
Run the sentence-transformers version of the search. This cell downloads a model of about 90 MB the first time. Score it on your test sheet. Your result will differ from the tuned word method.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = model.encode([p for n, p in passages], normalize_embeddings=True)

def retrieve3(question):
    qv = model.encode([question], normalize_embeddings=True)[0]
    scores = vectors @ qv
    i = int(scores.argmax())
    return passages[i], round(float(scores[i]), 2)

r3 = score(tests, retrieve3)
print((r3.policy == r3.got).sum(), 'of 12 right policy;', r3.answer_found.sum(), 'answers found')

In [ ]:
# r3 comes from the SentenceTransformer cell above
assert (r3.policy == r3.got).sum() >= 7, 'Fewer than 7 right policies. Encode every passage with normalize_embeddings=True, encode the question the same way, and take the best dot product'
print('Task 6 correct')

## Task 7. Let a small model answer (optional, Colab only)
Check the licence on the model card first. Then run the cell below. The model answers only from the passage that retrieval found. Compare three replies with the policy text.

In [ ]:
from transformers import pipeline
generate = pipeline('text-generation',
    model='Qwen/Qwen2.5-0.5B-Instruct')
RULE = 'Answer only from the passage. Else say so.'
def answer(question):
    (name, passage), s = retrieve2(question)
    text = f'Passage: {passage}\nQuestion: {question}'
    msgs = [{'role': 'system', 'content': RULE},
            {'role': 'user', 'content': text}]
    out = generate(msgs, max_new_tokens=60,
                   do_sample=False)
    return out[0]['generated_text'][-1]['content']
print(answer('How long do points last?'))

## Task 8. Record and reflect
Write two lines below. First, one entry for your use log: what you asked an assistant, what it gave, how you checked, and whether you kept it (write 'none used' if you used no assistant). Second, one sentence on the failure you found hardest to fix, and why. Do not paste any customer data into an assistant.